# Stage 4 — Topic interpretation and labeling

In [1]:
# Load saved assignments and text profiles only.

from pathlib import Path
import numpy as np
import pandas as pd

ARTIFACTS = Path("artifacts")  # notebook is located in project root
cluster_profiles = pd.read_csv(ARTIFACTS / "final_cluster_profiles.csv")
document_clusters = pd.read_csv(
    ARTIFACTS / "final_document_clusters.csv", dtype={"id": str}
)

print(f"Cluster profiles: {len(cluster_profiles):,}")
print(f"Assigned publications: {len(document_clusters):,}")

display(
    cluster_profiles.nlargest(8, "n_papers")[
        ["cluster_id", "n_papers", "centroid_norm", "top_terms"]
    ]
)

Cluster profiles: 160
Assigned publications: 75,110


,cluster_id,n_papers,centroid_norm,top_terms
34,34,1222,0.698970,"translation, machine translation, machine, mt,..."
76,76,1100,0.626458,"llms, models, language, large language, langua..."
15,15,1015,0.728135,"rag, retrieval, retrieval augmented, augmented..."
140,140,957,0.688662,"speech, tts, text, text speech, synthesis, mod..."
47,47,833,0.680589,"tuning, fine tuning, fine, models, parameter, ..."
149,149,796,0.730350,"reasoning, models, reasoning models, thinking,..."
19,19,792,0.661906,"reward, rl, reinforcement, reinforcement learn..."
40,40,787,0.612857,"data, synthetic, synthetic data, training, qua..."


## 1 Inspecting 8 largest final clusters

In [2]:
rng = np.random.default_rng(42)
largest_eight = cluster_profiles.nlargest(8, "n_papers")

for _, profile in largest_eight.iterrows():
    c = int(profile["cluster_id"])
    members = document_clusters.loc[document_clusters["cluster_id"] == c]
    random_docs = members.sample(n=min(3, len(members)), random_state=int(rng.integers(0, 2**31 - 1)))

    print("\n" + "=" * 70)
    print(f"CLUSTER {c} — {len(members):,} papers")
    print(f"Centroid norm: {profile['centroid_norm']:.4f}")
    print("=" * 70)
    print("\nTop terms:")
    print(profile["top_terms"])
    print("\nRepresentative titles:")
    for title in str(profile["representative_titles"]).split(" | ")[:2]:
        print("-", title)
    print("\nRandom titles:")
    for title in random_docs["clean_title"]:
        print("-", title)


CLUSTER 34 — 1,222 papers
Centroid norm: 0.6990

Top terms:
translation, machine translation, machine, mt, nmt, neural machine, translations, english, quality, neural, translation quality, languages

Representative titles:
- HW-TSC's Submission to the CCMT 2024 Machine Translation Tasks
- Evaluating the Impact of Verbal Multiword Expressions on Machine Translation

Random titles:
- EBBS: An Ensemble with Bi-Level Beam Search for Zero-Shot Machine Translation
- POMP: Probability-driven Meta-graph Prompter for LLMs in Low-resource Unsupervised Neural Machine Translation
- Evaluation of Chinese-English Machine Translation of Emotion-Loaded Microblog Texts: A Human Annotated Dataset for the Quality Assessment of Emotion Translation

CLUSTER 76 — 1,100 papers
Centroid norm: 0.6265

Top terms:
llms, models, language, large language, language models, large, models llms, llm, tasks, performance, human, capabilities

Representative titles:
- Can Language Models Recognize Convincing Arguments?


## 2. Targeted inspection of ambiguous clusters

In [3]:

import pandas as pd
from pathlib import Path

docs = pd.read_csv(
    Path("../artifacts/final_document_clusters.csv")
    if Path("../artifacts/final_document_clusters.csv").exists()
    else Path("artifacts/final_document_clusters.csv")
)

cluster_col = next(
    c for c in ["cluster_id", "cluster", "cluster_label", "label"]
    if c in docs.columns
)
title_col = next(
    c for c in ["title", "paper_title", "clean_title", "title_clean"]
    if c in docs.columns
)

clusters_to_review = [76, 140, 19, 40]

for cid in clusters_to_review:
    subset = docs.loc[docs[cluster_col] == cid]

    sample = subset.sample(
        n=min(8, len(subset)),
        random_state=2026
    )

    print(f"\n{'=' * 70}")
    print(f"CLUSTER {cid} — additional random inspection")
    print(f"{'=' * 70}")

    for i, title in enumerate(sample[title_col], 1):
        print(f"{i:2d}. {title}")



CLUSTER 76 — additional random inspection
 1. Distilling LLMs' Decomposition Abilities into Compact Language Models
 2. Can LLMs Compute with Reasons?
 3. Analyzing Large language models chatbots: An experimental approach using a probability test
 4. Stereotype Detection in LLMs: A Multiclass, Explainable, and Benchmark-Driven Approach
 5. The Trilemma of Truth in Large Language Models
 6. Large Language Models Understand Layout
 7. Continual Learning for Large Language Models: A Survey
 8. An Analysis and Mitigation of the Reversal Curse

CLUSTER 140 — additional random inspection
 1. Finding A Voice: Exploring the Potential of African American Dialect and Voice Generation for Chatbots
 2. A Study of Modeling Rising Intonation in Cantonese Neural Speech Synthesis
 3. Application of Knowledge Distillation to Multi-task Speech Representation Learning
 4. Does human speech follow Benford's Law?
 5. Self-Supervised Models of Speech Infer Universal Articulatory Kinematics
 6. Enhancing Sp